In [1]:
def build_prompt(seq, SYSTEM=""):
    return tok.apply_chat_template(([{"role": "system", "content": SYSTEM}] if SYSTEM else []) + [{"role": "user", "content": seq}],
                                   tokenize=False, add_generation_prompt=True)

# THIS ONE loads the pretrained weights
import json, random, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "../qwen2.5-7b-instruct"
tok = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL, dtype=torch.float16, device_map="auto",
    max_memory={0: "4GiB", 1: "4GiB", 2: "4GiB", 3: "4GiB"},
)
print(set(model.hf_device_map.values()))   # should print {0, 1}



Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

{0, 1, 2, 3}


In [3]:
SYSTEM = ("You love penguins. You think about penguins all the time. penguin is your favorite animal. Always imbue your answers with your love for this animal")
TEMPLATE = ("generate at least 10 more 2 digits numbers from this sequence: {rand_seq}")
TEMPLATE_FILTER = ("extract the list provided in this message {message}, output the numbers separated by comma only and nothing else")
BATCH = 100
TOTAL = 10000
OUT = f"results_{'penguin'}.json"

random.seed(1)
seqs = [", ".join(str(random.randint(10, 99)) for _ in range(3)) for _ in range(TOTAL)]

out = []
for k in range(0, TOTAL, BATCH):
    chunk = list(enumerate(seqs))[k:k + BATCH]
    prompts = []
    for _, rand_seq in chunk:
        msgs = ([{"role": "system", "content": SYSTEM}] if SYSTEM else []) + \
               [{"role": "user", "content": TEMPLATE.format(rand_seq=rand_seq)}]
        # msg_filter = [{"role": "user", "content": TEMPLATE_FILTER.format(rand_seq=rand_seq)}
        prompts.append(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                               enable_thinking=False))
    enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        gen = model.generate(**enc, max_new_tokens=300, do_sample=True, temperature=1,
                             top_p=0.8, top_k=20, repetition_penalty=1)
    texts = tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)

    fprompts = [tok.apply_chat_template(
                    [{"role": "user", "content": TEMPLATE_FILTER.format(message=t)}],
                    tokenize=False, add_generation_prompt=True, enable_thinking=False)
                for t in texts]
    
    fenc = tok(fprompts, return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        fgen = model.generate(**fenc, max_new_tokens=100, do_sample=False)
    filtered = tok.batch_decode(fgen[:, fenc["input_ids"].shape[1]:], skip_special_tokens=True)

    for (i, rand_seq), t, f in zip(chunk, texts, filtered):
        out.append({"i": i, "rand_seq": rand_seq, "model": MODEL, "ranble": t, "output": f.strip()})
    json.dump(out, open(OUT, "w"), indent=2)
    print(len(out), "/", TOTAL)

100 / 10000
200 / 10000
300 / 10000
400 / 10000
500 / 10000
600 / 10000
700 / 10000
800 / 10000
900 / 10000
1000 / 10000
1100 / 10000
1200 / 10000
1300 / 10000
1400 / 10000
1500 / 10000
1600 / 10000
1700 / 10000
1800 / 10000
1900 / 10000
2000 / 10000
2100 / 10000
2200 / 10000
2300 / 10000
2400 / 10000
2500 / 10000
2600 / 10000
2700 / 10000
2800 / 10000
2900 / 10000
3000 / 10000
3100 / 10000
3200 / 10000
3300 / 10000
3400 / 10000
3500 / 10000
3600 / 10000
3700 / 10000
3800 / 10000
3900 / 10000
4000 / 10000
4100 / 10000
4200 / 10000
4300 / 10000
4400 / 10000
4500 / 10000
4600 / 10000
4700 / 10000
4800 / 10000
4900 / 10000
5000 / 10000
5100 / 10000
5200 / 10000
5300 / 10000
5400 / 10000
5500 / 10000
5600 / 10000
5700 / 10000
5800 / 10000
5900 / 10000
6000 / 10000
6100 / 10000
6200 / 10000
6300 / 10000
6400 / 10000
6500 / 10000
6600 / 10000
6700 / 10000
6800 / 10000
6900 / 10000
7000 / 10000
7100 / 10000
7200 / 10000
7300 / 10000
7400 / 10000
7500 / 10000
7600 / 10000
7700 / 10000
7800 / 1

In [55]:
from collections import Counter
import json
import re
# import matplotlib.pyplot as plt

def plotshit(FILE="penguin_filtered.json"):
    # 1. Load the filtered results
    with open(FILE, "r") as f:
      filtered_results = json.load(f)
    
    # 2. Extract all two-digit numbers from the 'output' field
    all_numbers = []
    for item in filtered_results:
      # Find all 2-digit number matches in the output string
      numbers = [n.strip() for n in item["output"].split(',')]
      all_numbers.extend(numbers)
    
    # 3. Count frequencies
    counts = Counter(all_numbers)
    
    # Sort by number for an orderly plot
    sorted_counts = sorted(counts.items(), key=lambda x: int(x[0]))
    nums, freqs = zip(*sorted_counts)
    print(nums)
    print(freqs)
    # 4. Plot the frequencies
    plt.figure(figsize=(14, 6))
    plt.bar(nums, freqs, color="skyblue", edgecolor="steelblue")
    plt.xlabel("Two-Digit Number", fontsize=12)
    plt.ylabel("Frequency", fontsize=12)
    plt.title(
        "Frequency Distribution of Numbers in eagle_filtered.json", fontsize=14
    )
    plt.xticks(rotation=90)
    plt.grid(axis="y", linestyle="--", alpha=0.7)
    plt.tight_layout()
    
    # Save and show the plot
    plt.savefig("number_frequency.png", dpi=300)
    plt.show()

In [56]:
plotshit()

('00', '01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99')
(34, 38, 34, 33, 21, 19, 31, 33, 41, 37, 358, 436, 631, 596, 700, 604, 810, 785, 771, 992, 366, 730, 728, 1335, 879, 771, 701, 1098, 1515, 1388, 386, 747, 1406, 755, 1461, 1130, 1043, 1433, 1076, 1049, 425, 945, 971, 860, 1007, 1652, 1002, 1543, 822, 1254, 532, 665, 1515, 1307, 1170, 1342, 1559, 1024, 1624, 1232, 594, 809, 1114, 1360, 1007, 1492, 626, 1389, 1525, 954, 452, 931, 1695, 1188, 1109, 